# 02b — GLM Pricer (owns its calculation)
This notebook IS the GLM method: the CALC cell below computes the premium and registers it. `pricing.py` only connects (`quote`/`api_quote`). Hub `02_pricing.ipynb` and `03_main.ipynb` replay this exact cell via the loader — one source of truth.

In [ ]:
import os, sys
ROOT = os.path.abspath('')
if ROOT not in sys.path: sys.path.insert(0, ROOT)

import pandas as pd
from voltvision import load_template, describe_pricer, resolve_cards, api_quote
from voltvision import io
from voltvision.simulate import simulate_book
REGIME = 'glm'


In [ ]:
# CALC — owned by this notebook. The GLM method owns its card, its feature
# list and its training setup. Scenarios may override any declared parameter:
#   "pricing": {"glm": {"expense_loading": 1.8}}
from voltvision.assumptions import deep_merge
from voltvision.ml import fit_frequency, severity_table, severity_by_row, encode_features
from voltvision.pricing import register_pricer
from voltvision.simulate import simulate_book

CARD = {
    'expense_loading':    {'default': 1.5,  'unit': 'multiplier',   'note': 'pure-premium loading; raise -> lower LR'},
    'risk_step':          {'default': 1.1,  'unit': 'per flag',     'note': 'multiplier per true risk flag'},
    'glm_alpha':          {'default': 1e-3, 'unit': 'L2 penalty',   'note': 'Poisson regularization'},
    'train_frac':         {'default': 0.6,  'unit': 'fraction',     'note': 'share of training rows used to fit'},
    'train_seed':         {'default': 7,    'unit': 'seed',         'note': 'train-split seed'},
    'train_book_seed':    {'default': 42,   'unit': 'seed or null', 'note': 'separate historical book; null = legacy in-sample'},
    'train_window_years': {'default': 5,    'unit': 'years',        'note': 'training horizon, one period before the priced cohort'},
    'train_vehicle':      {'default': None, 'unit': 'share dict or null', 'note': 'training fleet mix; null = the training world vehicle_mix'},
    'train_dgp':          {'default': {},   'unit': 'engine overrides', 'note': 'extra training-world assumptions; {} = base template only, scenario engine patches NOT applied'}
}

# Features this method prices on (the method owns its feature list).
FEATURES = ['DRIVER_AGE', 'CAR_AGE', 'NCD_LEVEL', 'VEHICLE_TYPE',
            'COVERAGE_TYPE', 'FLOOD_RISK', 'THEFT_RISK', 'REGION']


def training_history(card, cfg, base_cfg):
    """Training dataset: generated from the BASE template assumptions (scenario
    DGP patches do NOT flow in), one window earlier than the priced cohort.

    Why: historical experience is fixed. If the training book were generated
    under the scenario's stressed DGP, the model would pre-price a shock it
    never lived through and stress tests would show fake stability at the cost
    of inflated premiums. train_dgp adds explicit training-world overrides;
    train_book_seed = null reverts to legacy in-sample training.
    """
    world = base_cfg if base_cfg is not None else cfg
    train_cfg = deep_merge(world, card.train_dgp or {})
    train_cfg['cohort_year'] = cfg['cohort_year'] - card.train_window_years
    vehicle = card.train_vehicle or train_cfg['vehicle_mix']
    return simulate_book(train_cfg, vehicle, card.train_book_seed,
                         n_years=card.train_window_years, cache=True)


def price_glm(book, card, cfg, base_cfg=None):
    """Standard method interface: (book, card, cfg, base_cfg) -> + FINAL_PREMIUM_SST."""
    o = book.copy()
    src = training_history(card, cfg, base_cfg) if card.train_book_seed is not None else o
    training_rows = src[src['COHORT_YEAR'] == src['SIM_YEAR']].sample(
        frac=card.train_frac, random_state=card.train_seed)
    model = fit_frequency(training_rows, FEATURES, card.glm_alpha)
    sev, covsev = severity_table(src)
    premium = (model.predict(encode_features(o, FEATURES))
               * severity_by_row(o, sev, covsev) * card.expense_loading)
    premium = (premium
               * card.risk_step ** o['FLOOD_RISK'].values.astype(int)
               * card.risk_step ** o['THEFT_RISK'].values.astype(int))
    return o.assign(FINAL_PREMIUM_SST=premium.round(2))


register_pricer('glm', price_glm, card=CARD, info={
    'label': 'GLM',
    'color': '#f59e0b',
    'formula': ('Poisson freq (features above) x avg severity x expense_loading '
                'x risk_step^flags'),
})
print('glm calc registered')


## Rule sheet (`describe_pricer('glm')`)
| Piece | Rule |
|---|---|
| Frequency | PoissonRegressor (`alpha=glm_alpha`) on DRIVER_AGE, CAR_AGE, NCD_LEVEL, VEHICLE_TYPE, COVERAGE_TYPE, FLOOD_RISK, THEFT_RISK, REGION; trained on `train_frac` of rows where COHORT_YEAR == SIM_YEAR (`train_seed`) |
| Severity | mean CLAIM_AMOUNT per CLAIM_COUNT by (COVERAGE_TYPE, VEHICLE_TYPE), fallback to by COVERAGE_TYPE |
| Premium | `freq × severity × expense_loading × risk_step^flags` (SST not applied to GLM leg) |
| Live params | declared in `CARD` inside the CALC cell (`expense_loading`, `train_book_seed`, ...) — override per scenario via `"pricing": {"glm": {...}}` |


In [ ]:
cfg = load_template()
print(describe_pricer(REGIME)['label'], '- declared parameters:')
display(pd.DataFrame(describe_pricer(REGIME)['params']))
cards = resolve_cards(cfg)
print('resolved cards for:', list(cards))


## Request (simulated book in)

In [ ]:
SC, SEED_RUN = 'MIX', 0   # any scenario/seed written by run_scenarios.py
try:
    raw = io.sim_book(io.load_result(SC, SEED_RUN))
    print(f'loaded shared/results/{SC}_s{SEED_RUN}: {len(raw)} rows')
except FileNotFoundError:
    print('no result yet - quick inline sim (run `python run_scenarios.py` for real books)')
    quick = {**load_template(), 'n': 1000, 'n_years': 2}
    raw = simulate_book(quick, quick['vehicle_mix'], seed=0, n_years=2)


## Response (API format: regime + label + card + metrics + priced book)

In [ ]:
resp = api_quote(raw, REGIME, cards[REGIME], cfg)
print('regime:', resp['regime'], '|', resp['label'])
display(pd.DataFrame([resp['metrics']]))
display(resp['book'][['POLID', 'COVERAGE_TYPE', 'VEHICLE_TYPE', 'CLAIM_COUNT',
    'CLAIM_AMOUNT', 'FINAL_PREMIUM_SST']].head())


## Notes
- Trains out-of-sample by default (seed 42, window 2021–2025); `train_book_seed=None` restores the legacy in-sample fit.
- Siblings: `02a_tariff.ipynb`, `02c_telem.ipynb` (same request/response shape).